# Grammar Testing for Ancient Greek
To be more specific, this analysis was conducted using a subset of Ancient Greek (see README for the exact vocabulary used). It is also important to note that the use of transliteration into the Latin alphabet is intentional, as it allows the computational process to run more efficiently and effectively compared to using the original Greek script.

I would like to acknowledge that, for a more realistic natural language processing system for Ancient Greek, the program would need to handle additional linguistic features such as accents, inflection, and the language’s unique symbols. Unfortunately, developing such a system is beyond the scope of my current capabilities.

That said, for the purposes of this project, the use of a simplified subset of words and their analysis was sufficient for me to develop a conceptual understanding of the topics covered in class.

In [1]:
import nltk
from nltk import CFG
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\marti\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
# defining the context-free grammar
grammar_base = CFG.fromstring("""
    S -> S CONJ S
    S -> NP VP

    NP -> NP CONJ NP
    NP -> NP PP
    NP -> DET N
    NP -> N

    VP -> VP CONJ VP
    VP -> VP PP
    VP -> V
    VP -> V NP

    PP -> P NP

    DET -> 'ho' | 'ton' | 'to'
    N -> 'aner' | 'doulon' | 'agro' | 'oikon'
    V -> 'blepei' | 'pherei' | 'legei'
    P -> 'en' | 'pros'
    CONJ -> 'kai'
""")

In [3]:
# defining the elminated ambiguity grammar

grammar_ambiguity = CFG.fromstring("""

    S -> S CONJ S
    S -> NP VP

    NP -> NP CONJ NP
    NP -> DET N
    NP -> N

    VP -> VP CONJ VP
    VP -> VP PP
    VP -> V
    VP -> V NP

    PP -> P NP 
                              
    DET -> 'ho' | 'ton' | 'to'
    N -> 'aner' | 'doulon' | 'agro' | 'oikon'
    V -> 'blepei' | 'pherei' | 'legei'
    P -> 'en' | 'pros'
    CONJ -> 'kai'
                              
""")

                        

In [4]:
# defining the elminated left recursion grammar

grammar_lf = CFG.fromstring("""
    S -> NP VP
    S -> NP VP CONJ S

    NP -> DET N
    NP -> N
    NP -> DET N CONJ NP
    NP -> N CONJ NP

    VP -> V
    VP -> V NP
    VP -> V PP
    VP -> V NP PP
    VP -> V CONJ VP
    VP -> V NP CONJ VP

    PP -> P NP

    DET -> 'ho' | 'ton' | 'to'
    N -> 'aner' | 'doulon' | 'agro' | 'oikon'
    V -> 'blepei' | 'pherei' | 'legei'
    P -> 'en' | 'pros'
    CONJ -> 'kai'
""")

In [5]:
# creating the parsers for each grammars
parser_base = nltk.ChartParser(grammar_base)

parser_ambiguity = nltk.ChartParser(grammar_ambiguity)

parser_lf = nltk.ChartParser(grammar_lf)


In [6]:
# define the experimental sentence
sentence = "ho aner pherei ton doulon en to agro"
# tokeniza via split
tokens = sentence.split()
tokens

['ho', 'aner', 'pherei', 'ton', 'doulon', 'en', 'to', 'agro']

In [7]:
# Parse the sentence with base grammer
for tree in parser_base.parse(tokens):
    tree.pretty_print()

                     S                             
      _______________|________                      
     |                        VP                   
     |                ________|_________            
     |               VP                 PP         
     |          _____|___            ___|___        
     NP        |         NP         |       NP     
  ___|___      |      ___|____      |    ___|___    
DET      N     V    DET       N     P  DET      N  
 |       |     |     |        |     |   |       |   
 ho     aner pherei ton     doulon  en  to     agro

                     S                             
      _______________|________                      
     |                        VP                   
     |          ______________|_____                
     |         |                    NP             
     |         |          __________|___            
     |         |         |              PP         
     |         |         |           ___|___        
  

In [8]:
# Parse the sentence with elminated ambiguity grammer
for tree in parser_ambiguity.parse(tokens):
    tree.pretty_print()

                     S                             
      _______________|________                      
     |                        VP                   
     |                ________|_________            
     |               VP                 PP         
     |          _____|___            ___|___        
     NP        |         NP         |       NP     
  ___|___      |      ___|____      |    ___|___    
DET      N     V    DET       N     P  DET      N  
 |       |     |     |        |     |   |       |   
 ho     aner pherei ton     doulon  en  to     agro



In [9]:
# Parse the sentence with elminated left recursion grammer
for tree in parser_lf.parse(tokens):
    tree.pretty_print()

                     S                             
      _______________|________                      
     |                        VP                   
     |          ______________|_________            
     |         |         |              PP         
     |         |         |           ___|___        
     NP        |         NP         |       NP     
  ___|___      |      ___|____      |    ___|___    
DET      N     V    DET       N     P  DET      N  
 |       |     |     |        |     |   |       |   
 ho     aner pherei ton     doulon  en  to     agro



# Takeaway
The base grammar produces two parse trees for the same sentence, demonstrating ambiguity. After eliminating ambiguity, only one parse tree is generated. After removing left recursion, the structure of the parse tree changes to include auxiliary nonterminals, making the grammar suitable for top-down parsing.

# Testing Certain Phrases with LL(1) Parser

In [10]:
# define sentences that are expected to be accpeted and rejected
accepted_sentences = [
    "ho aner blepei",
    "ho aner pherei ton doulon",
    "ho aner pherei ton doulon en to agro",
    "aner legei",
    "ho aner blepei kai aner legei"
]

rejected_sentences = [
    "blepei ho aner",
    "ho pherei aner",
    "ho aner en to agro",
    "kai ho aner blepei",
    "ho aner pherei ton"
]

In [11]:
# funtion to automate the parser test

def test_sentence(parser, sentence):
    tokens = sentence.split() # create tokens.
    trees = list(parser.parse(tokens)) # obtain a syntax tree; if any
    return len(trees) > 0 # return a false if not syntax tree is found


In [12]:
for s in accepted_sentences:
    print(f"'{s}' is", "ACCEPTED" if test_sentence(parser_lf, s) else "REJECTED")

for s in rejected_sentences:
    print(f"'{s}' is", "ACCEPTED" if test_sentence(parser_lf, s) else "REJECTED")



'ho aner blepei' is ACCEPTED
'ho aner pherei ton doulon' is ACCEPTED
'ho aner pherei ton doulon en to agro' is ACCEPTED
'aner legei' is ACCEPTED
'ho aner blepei kai aner legei' is ACCEPTED
'blepei ho aner' is REJECTED
'ho pherei aner' is REJECTED
'ho aner en to agro' is REJECTED
'kai ho aner blepei' is REJECTED
'ho aner pherei ton' is REJECTED
